<a href="https://colab.research.google.com/github/dhrubo404/Arush-Metals-Analyzer/blob/main/01_des_skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DES Skeleton

**EV Charging Allocation with Daisy-Chain Time-Sharing**

Goal: a working SimPy-based event simulator for the EV charging system, exercised on a trivial scenario (1 station, 1 charger of type L2s, FCFS), with passing smoke tests.

### Scope today

- Data classes for EV attributes and per-EV stats
- `ChargerPool` wrapper around `simpy.PreemptiveResource`
- `Station` aggregating pools (chains stubbed for later)
- `Simulator` that owns the env, runs arrivals, dispatches assignments, drives charging, collects stats
- `EventLog` for E1–E8 traces (debugging only; not the SimPy event queue)
- `Policy` interface with one concrete implementation: greedy first-fit
- Two smoke tests: basic FCFS, and deadline-during-charge

### Out of scope today (later days per `PLAN.md`)

- Daisy chains (class stubbed, not exercised) — Day 6–7
- Poisson arrivals + TOU prices (E8) — Day 4
- Multi-station policies, DP — Week 2
- Deadline penalty κ wired into a `J_i` cost

### Design notes

- SimPy processes per EV; one process generates arrivals.
- Chargers are `PreemptiveResource(capacity=n_{s,ℓ})`. Preemption isn't exercised today but the primitive is in place from day one (needed Day 6–7 for chain switches).
- Remaining energy `e_j` is recomputed at every event affecting an EV (start, complete, preempt) rather than tracked continuously — cleaner for SimPy's discrete-event model.
- Charging cost is integrated as `∫ p(t) π_ℓ rate dt` over each active interval. The integrator collapses to a midpoint multiplication when `p(t)` is flat (today). Day 4 replaces only the `_integrate_cost` body.
- The event log records `(t, event_type, ev_id, details)` — useful for asserting ordering in tests and for sanity-checking simulation traces.


In [ ]:
!pip install simpy


## 1. Imports and model-spec constants

Charger types, power outputs, and base prices are lifted directly from §2 of the model spec. The event taxonomy follows §6.


In [ ]:
import simpy
import random
import math
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Optional, Callable

# Charger types from §2 of the model spec
class ChargerType(Enum):
    L2s = "L2s"
    L2f = "L2f"
    DCm = "DCm"
    DCh = "DCh"

POWER_KW = {
    ChargerType.L2s: 7.0,
    ChargerType.L2f: 19.0,
    ChargerType.DCm: 150.0,
    ChargerType.DCh: 350.0,
}

BASE_PRICE = {  # $/kWh
    ChargerType.L2s: 0.15,
    ChargerType.L2f: 0.20,
    ChargerType.DCm: 0.40,
    ChargerType.DCh: 0.55,
}

# Event taxonomy from §6
class EventType(Enum):
    E1_ARRIVAL = auto()
    E2_SERVICE_START = auto()
    E3_SERVICE_COMPLETE = auto()
    E4_DEADLINE_EXPIRE = auto()
    E5_CHAIN_ADMISSION = auto()      # stubbed for Day 6–7
    E6_CHAIN_OCCUPANT_COMPLETE = auto()
    E7_CHAIN_SWITCH = auto()
    E8_PRICE_CHANGE = auto()         # wired Day 4


## 2. Data classes

- `EVAttributes` is the `(e_req, d, ρ)` tuple from §4.
- `EVStats` is the per-EV record collected over its lifetime — the raw data for the §9 metrics.
- `EventRecord` + `EventLog` give a trace of the discrete events (E1–E8) for debugging and test assertions. Distinct from SimPy's internal event queue.


In [ ]:
@dataclass
class EVAttributes:
    """Per §4 of the model spec: (e_req, d, rho)."""
    e_req: float       # kWh requested
    deadline: float    # hours, relative to arrival
    rho_min: float     # min acceptable charger power (kW)

@dataclass
class EVStats:
    """Per-EV record collected at departure."""
    ev_id: int
    arrival_time: float
    connect_time: Optional[float] = None       # τ_conn_i
    depart_time: Optional[float] = None        # τ_depart_i
    e_req: float = 0.0
    e_delivered: float = 0.0
    energy_unmet: float = 0.0
    charging_cost: float = 0.0
    wait_time: float = 0.0                     # W_i = τ_conn - τ_arr
    deadline_met: bool = True
    assigned_to: Optional[str] = None

@dataclass
class EventRecord:
    t: float
    etype: EventType
    ev_id: Optional[int]
    details: dict = field(default_factory=dict)

class EventLog:
    """Records E1–E8 firings for inspection and test assertions.
    This is NOT the SimPy event queue; it's a log."""
    def __init__(self):
        self.records: list = []

    def log(self, t, etype, ev_id=None, **details):
        self.records.append(EventRecord(t=t, etype=etype, ev_id=ev_id, details=details))

    def filter(self, etype=None, ev_id=None):
        out = self.records
        if etype is not None:
            out = [r for r in out if r.etype == etype]
        if ev_id is not None:
            out = [r for r in out if r.ev_id == ev_id]
        return out


## 3. ChargerPool

A thin wrapper around `simpy.PreemptiveResource(capacity=n)`. The capacity matches `n_{s,ℓ}` from §3. Preemption isn't exercised in Day 3 but the primitive is here so Day 6–7 chain switches don't need a rewrite.

- `resource.queue` — SimPy's internal pending requests (read-only for our purposes)
- `resource.users` — currently-holding processes


In [ ]:
class ChargerPool:
    """Wraps a SimPy PreemptiveResource representing n_{s,ℓ} units of charger
    type ℓ at station s. FCFS by default; preemption priorities come into
    play when the lower-level chain scheduler switches occupants (Day 6–7).
    """
    def __init__(self, env, station_id, charger_type, capacity):
        self.env = env
        self.station_id = station_id
        self.charger_type = charger_type
        self.capacity = capacity
        self.resource = simpy.PreemptiveResource(env, capacity=capacity)

    @property
    def power_kw(self): return POWER_KW[self.charger_type]

    @property
    def base_price(self): return BASE_PRICE[self.charger_type]

    def label(self): return f"{self.station_id}/{self.charger_type.value}"

    def queue_length(self): return len(self.resource.queue)

    def in_service(self): return len(self.resource.users)


## 4. DaisyChain (stub)

A chain has `k_c` connection slots but only ONE underlying charger of type `ℓ_c` — exactly one occupant is active at a time. The lower-level scheduler (FCFS/EDF/LRF/RR/DP) decides which occupant gets the active slot.

Day 3 leaves this as a placeholder. State fields mirror §5 so the data model is in place for Day 6–7.


In [ ]:
class DaisyChain:
    """Placeholder. See §5 of the model spec."""
    def __init__(self, env, station_id, chain_id, charger_type, capacity):
        self.env = env
        self.station_id = station_id
        self.chain_id = chain_id
        self.charger_type = charger_type
        self.capacity = capacity
        # State (§5): occupants O_{s,c}, active a_{s,c}, queue q_{s,c}
        self.occupants: list = []
        self.active = None
        self.queue: list = []

    def label(self): return f"{self.station_id}/chain:{self.chain_id}"


## 5. Station

Bag of pools (one per charger type) and chains. Matches the per-station topology in §3.


In [ ]:
class Station:
    """Holds the ChargerPools and DaisyChains at one site (§3)."""
    def __init__(self, env, station_id):
        self.env = env
        self.station_id = station_id
        self.pools: dict = {}
        self.chains: list = []

    def add_pool(self, charger_type, capacity):
        self.pools[charger_type] = ChargerPool(self.env, self.station_id,
                                               charger_type, capacity)

    def add_chain(self, chain_id, charger_type, capacity):
        self.chains.append(DaisyChain(self.env, self.station_id, chain_id,
                                      charger_type, capacity))

    def available_pool_types(self):
        return list(self.pools.keys())


## 6. Assignment policy interface

Every upper-level policy plugs in via `AssignmentPolicy.assign`. For Day 3 we only have one — a trivial first-fit greedy that filters on the `P_ℓ ≥ ρ_i` feasibility constraint from §7 and tie-breaks on combined queue + in-service occupancy.

Day 4 adds greedy-cost, greedy-delay, and weighted variants. The rollout-based DP (Day 9–10) will be just another `AssignmentPolicy` subclass.


In [ ]:
class AssignmentPolicy:
    """Upper-level policy: given an arriving EV and the current system state,
    return a ChargerPool (or None if no feasible assignment)."""
    def assign(self, ev_attrs, stations):
        raise NotImplementedError

class GreedyFirstFit(AssignmentPolicy):
    """Picks the first feasible pool (P_ℓ >= ρ_i), tie-breaking by current
    queue + in-service occupancy. Trivial baseline."""
    def assign(self, ev_attrs, stations):
        candidates = [pool for s in stations for pool in s.pools.values()
                      if pool.power_kw >= ev_attrs.rho_min]
        if not candidates:
            return None
        candidates.sort(key=lambda p: p.queue_length() + p.in_service())
        return candidates[0]


## 7. Simulator

The orchestrator. Owns:

- the `simpy.Environment` (clock + event queue)
- the `random.Random` (reproducibility from a single seed)
- the stations and their pools/chains
- the event log and stats list
- the price function `p(t)` (flat for Day 3; TOU for Day 4)
- the active assignment policy

The EV lifecycle is one SimPy process per EV. It:

1. Asks the policy for a pool assignment (or departs immediately if infeasible).
2. Races a charger request against the deadline timer.
3. If the request wins, charges for the computed duration (or until the deadline, whichever comes first).
4. Finalizes stats and releases the slot.

**The `try / except / finally` block is the key correctness mechanism.** It guarantees the resource is released regardless of which path the EV exits on (normal completion, deadline cut-off mid-charge, or future preemption events). Without the `finally`, a deadline cut-off would leak the slot — caught by smoke test 2 below.


In [ ]:
class Simulator:
    def __init__(self, seed=42, price_fn=None):
        self.env = simpy.Environment()
        self.rng = random.Random(seed)
        self.stations: list = []
        self.event_log = EventLog()
        self.stats: list = []
        # Default price multiplier p(t) = 1 (flat). Day 4 replaces this.
        self.price_fn = price_fn if price_fn is not None else (lambda t: 1.0)
        self._next_ev_id = 0
        self.policy: AssignmentPolicy = GreedyFirstFit()

    # ---- topology ----
    def add_station(self, station_id):
        s = Station(self.env, station_id)
        self.stations.append(s)
        return s

    # ---- arrivals ----
    def schedule_deterministic_arrivals(self, arrivals):
        """For testing: a fixed list of (time, attrs) pairs."""
        self.env.process(self._deterministic_arrival_proc(arrivals))

    def _deterministic_arrival_proc(self, arrivals):
        for t, attrs in arrivals:
            delay = t - self.env.now
            if delay > 0:
                yield self.env.timeout(delay)
            self._spawn_ev(attrs)

    def _spawn_ev(self, attrs):
        ev_id = self._next_ev_id
        self._next_ev_id += 1
        self.event_log.log(self.env.now, EventType.E1_ARRIVAL, ev_id,
                           e_req=attrs.e_req, deadline=attrs.deadline,
                           rho_min=attrs.rho_min)
        self.env.process(self._ev_lifecycle(ev_id, attrs))

    # ---- per-EV lifecycle ----
    def _ev_lifecycle(self, ev_id, attrs):
        arrival_time = self.env.now
        stats = EVStats(ev_id=ev_id, arrival_time=arrival_time, e_req=attrs.e_req)

        # Upper-level decision (E1 → assignment)
        pool = self.policy.assign(attrs, self.stations)
        if pool is None:
            stats.depart_time = self.env.now
            stats.energy_unmet = attrs.e_req
            stats.deadline_met = False
            self.stats.append(stats)
            self.event_log.log(self.env.now, EventType.E4_DEADLINE_EXPIRE, ev_id,
                               reason="no_feasible_pool")
            return
        stats.assigned_to = pool.label()

        # Race charger request against deadline
        req = pool.resource.request(priority=0)
        deadline_event = self.env.timeout(attrs.deadline)
        result = yield req | deadline_event

        if req not in result:
            # Deadline fired before charger granted
            if req.triggered:
                pool.resource.release(req)
            else:
                req.cancel()
            stats.depart_time = self.env.now
            stats.wait_time = self.env.now - arrival_time
            stats.energy_unmet = attrs.e_req
            stats.deadline_met = False
            self.stats.append(stats)
            self.event_log.log(self.env.now, EventType.E4_DEADLINE_EXPIRE, ev_id,
                               reason="deadline_in_queue")
            return

        # Got the slot — service starts
        connect_time = self.env.now
        stats.connect_time = connect_time
        stats.wait_time = connect_time - arrival_time
        self.event_log.log(connect_time, EventType.E2_SERVICE_START, ev_id,
                           pool=pool.label())

        service_duration = attrs.e_req / pool.power_kw
        time_to_deadline = attrs.deadline - (connect_time - arrival_time)

        try:
            if service_duration <= time_to_deadline:
                yield self.env.timeout(service_duration)
                self._finalize_ev(ev_id, stats, pool, attrs.e_req, completed=True)
            else:
                # Deadline hits mid-charge: charge what we can
                yield self.env.timeout(time_to_deadline)
                e_delivered = pool.power_kw * time_to_deadline
                self._finalize_ev(ev_id, stats, pool, e_delivered, completed=False)
        except simpy.Interrupt:
            # Preempted (chain switch in later days)
            elapsed = self.env.now - connect_time
            e_delivered = pool.power_kw * elapsed
            self._finalize_ev(ev_id, stats, pool, e_delivered, completed=False)
        finally:
            # Always release the slot, whatever path we took out
            if req in pool.resource.users:
                pool.resource.release(req)

    def _finalize_ev(self, ev_id, stats, pool, e_delivered, completed):
        cost = self._integrate_cost(stats.connect_time, self.env.now, pool, e_delivered)
        stats.depart_time = self.env.now
        stats.e_delivered = e_delivered
        stats.energy_unmet = max(0.0, stats.e_req - e_delivered)
        stats.charging_cost = cost
        stats.deadline_met = completed
        self.stats.append(stats)
        if completed:
            self.event_log.log(self.env.now, EventType.E3_SERVICE_COMPLETE, ev_id)
        else:
            self.event_log.log(self.env.now, EventType.E4_DEADLINE_EXPIRE, ev_id,
                               reason="deadline_in_service")

    def _integrate_cost(self, t_start, t_end, pool, e_delivered):
        """∫ p(t) π_ℓ rate dt over the active interval.
        Day 3 with flat price: collapses to π_ℓ · p(midpoint) · e_delivered.
        Day 4 refines this with piecewise TOU breakpoints."""
        if t_end <= t_start:
            return 0.0
        t_mid = 0.5 * (t_start + t_end)
        return e_delivered * pool.base_price * self.price_fn(t_mid)

    def run(self, until):
        self.env.run(until=until)


## 8. Smoke test 1 — basic FCFS

One station, one L2s charger, three deterministic arrivals. Verifies:

- Every EV produces a stats record and reaches `E3_SERVICE_COMPLETE`.
- FCFS order is preserved at the resource queue.
- Wait times match hand-computed values (this is the load-bearing FCFS check).
- Charging cost matches `e_req × π_ℓ` for the L2s base price.
- Energy balance: `e_delivered == e_req` for every EV.


In [ ]:
def smoke_test_basic():
    sim = Simulator(seed=42)
    s1 = sim.add_station("station1")
    s1.add_pool(ChargerType.L2s, capacity=1)

    arrivals = [
        (0.0, EVAttributes(e_req=7.0,  deadline=2.0, rho_min=0.0)),  # 1 hr at 7 kW
        (0.1, EVAttributes(e_req=14.0, deadline=4.0, rho_min=0.0)),  # 2 hr at 7 kW
        (0.2, EVAttributes(e_req=3.5,  deadline=5.0, rho_min=0.0)),  # 0.5 hr at 7 kW
    ]
    sim.schedule_deterministic_arrivals(arrivals)
    sim.run(until=10.0)

    # All three EVs went through
    assert len(sim.stats) == 3, f"expected 3 stats, got {len(sim.stats)}"

    # Event presence
    starts = sim.event_log.filter(etype=EventType.E2_SERVICE_START)
    completes = sim.event_log.filter(etype=EventType.E3_SERVICE_COMPLETE)
    assert len(starts) == 3
    assert len(completes) == 3

    # FCFS order
    assert [r.ev_id for r in starts] == [0, 1, 2]

    # EV 0: arrives 0.0, charges 1 hr, completes 1.0
    ev0 = next(s for s in sim.stats if s.ev_id == 0)
    assert math.isclose(ev0.connect_time, 0.0)
    assert math.isclose(ev0.depart_time, 1.0)
    assert math.isclose(ev0.wait_time, 0.0)
    assert math.isclose(ev0.charging_cost, 7.0 * 0.15)

    # EV 1: waits until 1.0, charges 2 hr
    ev1 = next(s for s in sim.stats if s.ev_id == 1)
    assert math.isclose(ev1.connect_time, 1.0)
    assert math.isclose(ev1.depart_time, 3.0)
    assert math.isclose(ev1.wait_time, 0.9)

    # EV 2: waits until 3.0, charges 0.5 hr
    ev2 = next(s for s in sim.stats if s.ev_id == 2)
    assert math.isclose(ev2.connect_time, 3.0)
    assert math.isclose(ev2.depart_time, 3.5)
    assert math.isclose(ev2.wait_time, 2.8)

    # All complete successfully, full energy delivered
    assert all(s.deadline_met for s in sim.stats)
    assert all(math.isclose(s.e_delivered, s.e_req) for s in sim.stats)

    print("Smoke test 1 (basic FCFS): PASSED")
    print(f"  EVs processed: {len(sim.stats)}")
    print(f"  Event log entries: {len(sim.event_log.records)}")
    for s in sim.stats:
        print(f"  EV {s.ev_id}: arrive={s.arrival_time:.2f} "
              f"connect={s.connect_time:.2f} depart={s.depart_time:.2f} "
              f"wait={s.wait_time:.2f} cost=${s.charging_cost:.4f}")

smoke_test_basic()


Smoke test 1 (basic FCFS): PASSED
  EVs processed: 3
  Event log entries: 9
  EV 0: arrive=0.00 connect=0.00 depart=1.00 wait=0.00 cost=$1.0500
  EV 1: arrive=0.10 connect=1.00 depart=3.00 wait=0.90 cost=$2.1000
  EV 2: arrive=0.20 connect=3.00 depart=3.50 wait=2.80 cost=$0.5250


## 9. Smoke test 2 — deadline-during-charge release

Verifies the bug fix from the lifecycle: when an EV's deadline expires mid-charge, the slot must be released so the next EV can use it. Without the `try / finally` block, the slot leaks and EV 1 hangs in the queue forever.

- EV 0: wants 7 kWh (1 hr at 7 kW) but only has a 0.5 hr deadline → cut off mid-charge with 3.5 kWh delivered, 3.5 kWh unmet.
- EV 1: arrives at t=0.6, must get the (now-freed) slot and complete by t=1.1.


In [ ]:
def smoke_test_deadline_during_charge():
    """Verifies slot release when an EV is cut off mid-charge.
    Without the `finally: release` block, EV 1 hangs forever."""
    sim = Simulator(seed=42)
    s1 = sim.add_station("station1")
    s1.add_pool(ChargerType.L2s, capacity=1)

    arrivals = [
        (0.0, EVAttributes(e_req=7.0, deadline=0.5, rho_min=0.0)),  # cut off
        (0.6, EVAttributes(e_req=3.5, deadline=2.0, rho_min=0.0)),  # needs the freed slot
    ]
    sim.schedule_deterministic_arrivals(arrivals)
    sim.run(until=10.0)

    ev0 = next(s for s in sim.stats if s.ev_id == 0)
    ev1 = next(s for s in sim.stats if s.ev_id == 1)

    # EV 0: 0.5 hr of charging at 7 kW = 3.5 kWh delivered, 3.5 kWh unmet
    assert math.isclose(ev0.e_delivered, 3.5)
    assert math.isclose(ev0.energy_unmet, 3.5)
    assert not ev0.deadline_met
    assert math.isclose(ev0.depart_time, 0.5)

    # EV 1: must have got the charger after EV 0's slot was released
    assert ev1.connect_time is not None, "EV 1 never got the charger — release bug!"
    assert math.isclose(ev1.connect_time, 0.6)
    assert math.isclose(ev1.depart_time, 1.1)
    assert ev1.deadline_met

    print("Smoke test 2 (deadline-during-charge release): PASSED")
    print(f"  EV 0: cut off at t=0.5, delivered {ev0.e_delivered} kWh, unmet {ev0.energy_unmet} kWh")
    print(f"  EV 1: connected at t={ev1.connect_time}, completed at t={ev1.depart_time}")

smoke_test_deadline_during_charge()


Smoke test 2 (deadline-during-charge release): PASSED
  EV 0: cut off at t=0.5, delivered 3.5 kWh, unmet 3.5 kWh
  EV 1: connected at t=0.6, completed at t=1.1


## 10. Day 3 deliverable status

**Done**

- Core data classes (`EVAttributes`, `EVStats`, `EventRecord`, `EventLog`)
- `ChargerPool` around `PreemptiveResource`
- `Station` aggregator
- `DaisyChain` stub (interface only)
- `AssignmentPolicy` interface + `GreedyFirstFit`
- `Simulator` with deterministic arrival hook
- Smoke test 1: basic FCFS — 1 station, 1 L2s, 3 EVs
- Smoke test 2: deadline-during-charge — verifies slot release

**Bug caught and fixed**

The original lifecycle leaked the resource slot if an EV was cut off mid-charge or preempted. Replaced explicit release with a `try / finally` block so every exit path releases. Smoke test 2 was added to exercise this.

**Known stubs / deferred**

- Daisy-chain admission/complete/switch (E5/E6/E7) — Day 6–7
- Poisson arrivals + TOU schedule (E8) — Day 4
- Multi-station, multi-type policies — Day 4–5
- Deadline penalty κ wired into a `J_i` cost — needed once policies optimize against it; the integration scaffold is in place

**Next session (Day 4)**

- Poisson arrival generator
- TOU price function with piecewise integration
- Multi-station topology matching §3 of the spec
- Heuristic baselines: greedy-cost, greedy-delay, weighted greedy
